In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from PIL import Image
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Define data preprocessing and augmentation
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # EfficientNet-B4 input size
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Custom Dataset for Siamese Network
class SiameseSkinDataset(Dataset):
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.classes = np.unique(labels)
        # Check class distribution
        self.class_counts = {cls: np.sum(np.array(labels) == cls) for cls in self.classes}
        print(f"Class distribution in dataset: {self.class_counts}")
        for cls, count in self.class_counts.items():
            if count < 2:
                print(f"Warning: Class '{cls}' has only {count} sample(s). At least 2 samples per class are required for Siamese pairs.")
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img1_path = self.image_paths[idx]
        img1_label = self.labels[idx]
        
        should_get_same_class = np.random.randint(0, 2)
        
        if should_get_same_class:
            same_class_indices = np.where(np.array(self.labels) == img1_label)[0]
            same_class_indices = np.setdiff1d(same_class_indices, [idx])  # Exclude the current index
            if len(same_class_indices) == 0:
                # If no other samples in the same class, fallback to a negative pair
                different_class_indices = np.where(np.array(self.labels) != img1_label)[0]
                img2_idx = np.random.choice(different_class_indices)
                img2_path = self.image_paths[img2_idx]
                label = 1  # Treat as a negative pair
            else:
                img2_idx = np.random.choice(same_class_indices)
                img2_path = self.image_paths[img2_idx]
                label = 0  # Same class (similar)
        else:
            different_class_indices = np.where(np.array(self.labels) != img1_label)[0]
            if len(different_class_indices) == 0:
                # If no different class samples, fallback to a positive pair (rare case)
                same_class_indices = np.where(np.array(self.labels) == img1_label)[0]
                same_class_indices = np.setdiff1d(same_class_indices, [idx])
                img2_idx = np.random.choice(same_class_indices)
                img2_path = self.image_paths[img2_idx]
                label = 0  # Treat as a positive pair
            else:
                img2_idx = np.random.choice(different_class_indices)
                img2_path = self.image_paths[img2_idx]
                label = 1  # Different class (dissimilar)
        
        img1 = Image.open(img1_path).convert('RGB')
        img2 = Image.open(img2_path).convert('RGB')
        
        if self.transform:
            img1 = self.transform(img1)
            img2 = self.transform(img2)
        
        return img1, img2, torch.tensor(label, dtype=torch.float32)

# Siamese Network with EfficientNet-B4 backbone
class SiameseEfficientNet(nn.Module):
    def __init__(self):
        super(SiameseEfficientNet, self).__init__()
        self.backbone = torchvision.models.efficientnet_b4(weights='IMAGENET1K_V1')
        self.backbone.classifier = nn.Identity()
        self.fc = nn.Sequential(
            nn.Linear(1792, 512),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(512, 256)
        )
    
    def forward_one(self, x):
        x = self.backbone(x)
        x = self.fc(x)
        return x
    
    def forward(self, input1, input2):
        output1 = self.forward_one(input1)
        output2 = self.forward_one(input2)
        return output1, output2

# Contrastive Loss for Siamese Network
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin
    
    def forward(self, output1, output2, label):
        euclidean_distance = nn.functional.pairwise_distance(output1, output2)
        loss_similar = (1 - label) * torch.pow(euclidean_distance, 2)
        loss_dissimilar = label * torch.pow(torch.clamp(self.margin - euclidean_distance, min=0.0), 2)
        loss = 0.5 * (loss_similar + loss_dissimilar).mean()
        return loss

# Load and prepare dataset
def load_dataset(dataset_path):
    image_paths = []
    labels = []
    
    if not os.path.exists(dataset_path):
        print(f"Dataset path {dataset_path} does not exist.")
        return image_paths, labels
    
    classes = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))]
    print(f"Found classes: {classes}")
    
    for cls in classes:
        cls_path = os.path.join(dataset_path, cls)
        print(f"Checking class directory: {cls_path}")
        for img_name in os.listdir(cls_path):
            if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
                image_paths.append(os.path.join(cls_path, img_name))
                labels.append(cls)
    
    print(f"Loaded {len(image_paths)} images with labels: {np.unique(labels)}")
    return image_paths, labels

# Filter dataset to ensure each class has at least 2 samples
def filter_dataset(image_paths, labels):
    # Count samples per class
    label_counts = pd.Series(labels).value_counts()
    print(f"Initial class distribution: {label_counts.to_dict()}")
    
    # Identify classes with at least 2 samples
    valid_classes = label_counts[label_counts >= 2].index.tolist()
    print(f"Classes with at least 2 samples: {valid_classes}")
    
    # Filter dataset
    filtered_indices = [i for i, label in enumerate(labels) if label in valid_classes]
    filtered_image_paths = [image_paths[i] for i in filtered_indices]
    filtered_labels = [labels[i] for i in filtered_indices]
    
    print(f"After filtering, {len(filtered_image_paths)} images remain.")
    return filtered_image_paths, filtered_labels

# Training function
def train_model(model, train_loader, criterion, optimizer, device, num_epochs=20):
    model.train()
    for epoch in range(num_epochs):
        running_loss = 0.0
        for i, (img1, img2, label) in enumerate(train_loader):
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            
            optimizer.zero_grad()
            output1, output2 = model(img1, img2)
            loss = criterion(output1, output2, label)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            
            if (i + 1) % 10 == 0:
                print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {running_loss/10:.4f}')
                running_loss = 0.0

# Evaluation function
def evaluate_model(model, test_loader, device):
    model.eval()
    predictions = []
    true_labels = []
    
    with torch.no_grad():
        for img1, img2, label in test_loader:
            img1, img2, label = img1.to(device), img2.to(device), label.to(device)
            output1, output2 = model(img1, img2)
            distance = nn.functional.pairwise_distance(output1, output2)
            pred = (distance < 1).float()
            predictions.extend(pred.cpu().numpy())
            true_labels.extend(label.cpu().numpy())
    
    accuracy = accuracy_score(true_labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(true_labels, predictions, average='binary')
    return accuracy, precision, recall, f1

# Main execution
def main():
    # Set device
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Load dataset
    dataset_path = '/kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train'  # Replace <dataset-name> with actual dataset name
    image_paths, labels = load_dataset(dataset_path)
    
    # Check for empty dataset
    if not image_paths:
        raise ValueError("No images found in the dataset. Check the dataset path and structure.")
    
    # Filter dataset to ensure each class has at least 2 samples
    image_paths, labels = filter_dataset(image_paths, labels)
    if not image_paths:
        raise ValueError("No classes with sufficient samples (at least 2) found in the dataset.")
    
    # Split dataset
    train_paths, test_paths, train_labels, test_labels = train_test_split(
        image_paths, labels, test_size=0.2, stratify=labels, random_state=42
    )
    print(f"Training set: {len(train_paths)} images, Test set: {len(test_paths)} images")
    
    # Verify training set class distribution
    train_label_counts = pd.Series(train_labels).value_counts()
    print(f"Training set class distribution: {train_label_counts.to_dict()}")
    for cls, count in train_label_counts.items():
        if count < 2:
            print(f"Warning: Class '{cls}' in training set has only {count} sample(s). This may cause issues.")
    
    # Create datasets and dataloaders
    train_dataset = SiameseSkinDataset(train_paths, train_labels, transform=transform)
    test_dataset = SiameseSkinDataset(test_paths, test_labels, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)
    
    # Initialize model, criterion, and optimizer
    model = SiameseEfficientNet().to(device)
    criterion = ContrastiveLoss(margin=2.0)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # Train model
    train_model(model, train_loader, criterion, optimizer, device, num_epochs=20)
    
    # Evaluate model
    accuracy, precision, recall, f1 = evaluate_model(model, test_loader, device)
    print(f'Test Accuracy: {accuracy:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1-Score: {f1:.4f}')

if __name__ == '__main__':
    main()

Using device: cuda
Found classes: ['HFMD', 'Monkeypox', 'Measles', 'Healthy', 'Chickenpox', 'Cowpox']
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train/HFMD
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train/Monkeypox
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train/Measles
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train/Healthy
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fold1_AUG/Train/Chickenpox
Checking class directory: /kaggle/input/mpox-skin-lesion-dataset-version-20-msld-v20/Augmented Images/Augmented Images/FOLDS_AUG/fo

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b4_rwightman-23ab8bcd.pth
100%|██████████| 74.5M/74.5M [00:01<00:00, 40.0MB/s]


Epoch [1/20], Step [10], Loss: 0.5310
Epoch [1/20], Step [20], Loss: 0.4955
Epoch [1/20], Step [30], Loss: 0.4690
Epoch [1/20], Step [40], Loss: 0.4779
Epoch [1/20], Step [50], Loss: 0.4748
Epoch [1/20], Step [60], Loss: 0.4865
Epoch [1/20], Step [70], Loss: 0.4185
Epoch [1/20], Step [80], Loss: 0.4288
Epoch [1/20], Step [90], Loss: 0.4373
Epoch [1/20], Step [100], Loss: 0.4087
Epoch [1/20], Step [110], Loss: 0.3854
Epoch [1/20], Step [120], Loss: 0.3823
Epoch [1/20], Step [130], Loss: 0.3884
Epoch [1/20], Step [140], Loss: 0.3493
Epoch [1/20], Step [150], Loss: 0.3544
Epoch [1/20], Step [160], Loss: 0.3491
Epoch [1/20], Step [170], Loss: 0.3763
Epoch [1/20], Step [180], Loss: 0.3376
Epoch [2/20], Step [10], Loss: 0.3157
Epoch [2/20], Step [20], Loss: 0.3627
Epoch [2/20], Step [30], Loss: 0.2988
Epoch [2/20], Step [40], Loss: 0.3253
Epoch [2/20], Step [50], Loss: 0.2818
Epoch [2/20], Step [60], Loss: 0.2890
Epoch [2/20], Step [70], Loss: 0.2663
Epoch [2/20], Step [80], Loss: 0.2511
Epo